In [93]:
%pip -q QuantLib numpy pandas scikit-learn matplotlib joblib

ERROR: unknown command "QuantLib"
Note: you may need to restart the kernel to use updated packages.


In [94]:
from pathlib import Path
from time import perf_counter
import json 
import joblib 
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd 
import QuantLib as ql 
from sklearn.dummy import DummyRegressor 
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor 
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import qmc
from sklearn.base import clone

In [95]:
seed = 42
n_samples = 1_000
S0 = 100.0 

OUTPUT = Path("surrogate models/experiment/1_baseline")
OUTPUT.mkdir(parents=True, exist_ok=True)

OUTPUT.mkdir(parents=True, exist_ok=True)
features = [
    "K", 
    "T_days",
    "r",
    "v0",
    "kappa",
    "theta",
    "sigma",
    "rho"
]
VALUATION_DATE = ql.Date(2, ql.January, 2026)
ql.Settings.instance().evaluationDate = VALUATION_DATE
DAY_COUNT = ql.Actual365Fixed()

In [96]:
def heston_quantlib_prices(
    K,
    T_days,
    r,
    v0,
    kappa,
    theta,
    sigma,
    rho,
    S0=S0
):
    maturity = VALUATION_DATE + int(T_days)
    risk_free = ql.YieldTermStructureHandle(ql.FlatForward(VALUATION_DATE, float(r), DAY_COUNT))
    dividend = ql.YieldTermStructureHandle(ql.FlatForward(VALUATION_DATE, 0.0, DAY_COUNT))
    spot = ql.QuoteHandle(ql.SimpleQuote(float(S0)))
    process = ql.HestonProcess(risk_free, dividend, spot, float(v0), float(kappa), float(theta), float(sigma), float(rho))
    engine = ql.AnalyticHestonEngine(ql.HestonModel(process))
    payoff = ql.PlainVanillaPayoff(ql.Option.Call, float(K))
    option = ql.VanillaOption(payoff, ql.EuropeanExercise(maturity))
    option.setPricingEngine(engine)
    return float(option.NPV())

In [97]:
def sample_parameters(n,seed):
    sampler = qmc.LatinHypercube(d=8, seed=seed)
    sample = sampler.random(n)
    lower = [0.6, 0.1, 0.0, 0.05, 0.01, 0.001, 0.001, -0.95]
    upper = [1.4, 1.4, 0.1, 0.5, 2.0, 0.5, 0.5, 0.0]
    values = qmc.scale(sample, lower, upper)
    df = pd.DataFrame(
        values, 
        columns=[
            "m",
            "T_years", 
            "r", 
            "v0", 
            "kappa", 
            "theta", 
            "sigma",
            "rho"
        ]
    )
    df["K"] = S0/df["m"]
    df["T_days"] = np.maximum(1,np.round(df["T_years"] * 365).astype(int))

    return df[features]

In [98]:
#mathematical validity test
rng_data = sample_parameters(n_samples, seed)
valid_test = (
    rng_data["K"].gt(0)
    & rng_data["T_days"].gt(0)
    & rng_data["v0"].gt(0)
    & rng_data["kappa"].gt(0)
    & rng_data["theta"].gt(0)
    & rng_data["sigma"].gt(0)
    & rng_data["rho"].between(-1,1)
    & np.isfinite(rng_data).all(axis=1)
)

print("Valid param sets: ", valid_test.sum())
print("Invalid param sets: ", (~valid_test).sum())

assert valid_test.all(), "Invalid Heston params found"


Valid param sets:  15000
Invalid param sets:  0


In [99]:
feller = (
    2 * rng_data["kappa"] * rng_data["theta"]
    >= rng_data["sigma"] ** 2
)

print("Feller Satisfied:", feller.sum())
print("Feller violated:", (~feller).sum())

print(
    "Feller satisfaction rate:",
    f"{feller.mean() * 100:.2f}%"
)

Feller Satisfied: 12725
Feller violated: 2275
Feller satisfaction rate: 84.83%


In [100]:
def label_with_quantlib(frame):
  prices = []
  start = perf_counter()
  for i, row in enumerate(frame[features].itertuples(index=False, name=None), 1):
    price = heston_quantlib_prices(*row)
    K, T_days, r, *_ = row
    lower_bound = max(0.0, S0 - K * np.exp(-r * T_days/365.0))
    if not np.isfinite(price) or not (lower_bound - 1e-5 <= price <= S0 + 1e-5):
      raise ValueError(f"Invalid QuantLib price at row {i}: {price}, input={row}")
    prices.append(price)
    if i % 1000 == 0: 
      print(f"Priced {i:,}/{len(frame):,} options")
  return np.asarray(prices), perf_counter() - start

In [101]:
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Generating Heston reference prices ***")
y, generation_secs = label_with_quantlib(rng_data)
data = rng_data.copy()
data["price"] = y 
data["feller"]=feller.to_numpy()
data.to_csv(DATA_DIR / "heston_prices_15k.csv", index=False)
print(f"Generated {len(data):,} labels in {generation_secs:.2f}s")
display(data.head())

Generating Heston reference prices ***
Priced 1,000/15,000 options
Priced 2,000/15,000 options
Priced 3,000/15,000 options
Priced 4,000/15,000 options
Priced 5,000/15,000 options
Priced 6,000/15,000 options
Priced 7,000/15,000 options
Priced 8,000/15,000 options
Priced 9,000/15,000 options
Priced 10,000/15,000 options
Priced 11,000/15,000 options
Priced 12,000/15,000 options
Priced 13,000/15,000 options
Priced 14,000/15,000 options
Priced 15,000/15,000 options
Generated 15,000 labels in 28.18s


,K,T_days,r,v0,kappa,theta,sigma,rho,price,feller
0,143.313277,472,0.095354,0.259799,1.413999,0.159716,0.409689,-0.832566,9.805618,True
1,113.637246,179,0.090824,0.159832,1.752093,0.267805,0.397391,-0.621884,8.471612,True
2,98.340302,285,0.018701,0.191461,1.200715,0.435251,0.230242,-0.330467,19.570676,True
3,109.487634,175,0.065177,0.241579,0.855730,0.404668,0.208659,-0.901358,11.581527,True
4,146.614104,197,0.060397,0.476294,1.617903,0.387443,0.402588,-0.078006,7.882521,True


In [102]:
X_train, X_rest, y_train, y_rest = train_test_split(
    data[features],
    y,
    test_size=0.30,
    random_state=seed
)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest,
    y_rest,
    test_size=0.50,
    random_state=seed
)

In [103]:
models = {
    "LinearRegression": make_pipeline(StandardScaler(), LinearRegression()),
    "MLP": make_pipeline(
        StandardScaler(), 
        MLPRegressor(
            hidden_layer_sizes=(128,128,128),
            activation="relu", 
            solver="adam",
            learning_rate_init = 1e-3,
            max_iter=400, 
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
            random_state = seed,
          ),
        ),
}

In [104]:
rows = []
prediction_table = X_test.reset_index(drop=True).copy()
prediction_table["actual_prices"] = np.asarray(y_test) 
bench_X = X_test.iloc[: min(1000, len(X_test))]
t0 = perf_counter()
for row in bench_X.itertuples(index=False, name=None):
  heston_quantlib_prices(*row)
t_quantlib = perf_counter() - t0

for name, model in models.items():
  t0 = perf_counter()
  model.fit(X_train, y_train)
  train_time = perf_counter() - t0
  pred_val = model.predict(X_val)
  pred_test = model.predict(X_test)
  model.predict(bench_X)
  timings = []
  for _ in range(10):
    t0 = perf_counter()
    model.predict(bench_X)
    timings.append(perf_counter() - t0)
  ml_time = float(np.median(timings))
  rows.append({
      "model": name,
      "val_MAE": mean_absolute_error(y_val, pred_val),
      "test_MAE": mean_absolute_error(y_test, pred_test),
      "test_RMSE": np.sqrt(mean_squared_error(y_test, pred_test)),
      "test_max_abs_error": float(np.max(np.abs(y_test - pred_test))),
      "test_median_abs_error": float(np.median(np.abs(np.asarray(y_test) - pred_test))),
      "test_p95": float(np.percentile(np.abs(np.asarray(y_test) - pred_test), 95)),
      "train_time": train_time,
      "quantlib_time": t_quantlib,
      "ml_time_batch": ml_time,
      "batch_speedup": t_quantlib / ml_time 
  })
  prediction_table[f"{name}_prediction"] = pred_test
  joblib.dump(model, OUTPUT / f"{name}.joblib")

In [105]:
result = pd.DataFrame(rows)
print("\nResults:\n", result.to_string(index=False))
result.to_csv(OUTPUT / "results.csv", index=False)
prediction_table.to_csv(OUTPUT / "test_predictions.csv", index=False)
plt.figure(figsize=(10,10))
plt.scatter(
    y_test, 
    prediction_table["MLP_prediction"],
    alpha=0.35,
    s=10
)
plt.plot(
    [0,S0], 
    [0,S0],
    "k--"
)
plt.xlim(0, max(y_test.max(), prediction_table["MLP_prediction"].max()) * 1.05)
plt.ylim(0, max(y_test.max(), prediction_table["MLP_prediction"].max()) * 1.05)
plt.xlabel("QuantLib references price")
plt.ylabel("MLP Predicted prices")
plt.title("Heston surrogate: test set")
plt.tight_layout()
plt.savefig(OUTPUT / "heston_mlp_test.png", dpi=180)
plt.close()


Results:
            model  val_MAE  test_MAE  test_RMSE  test_max_abs_error  test_median_abs_error  test_p95  train_time  quantlib_time  ml_time_batch  batch_speedup
LinearRegression 2.500541  2.459975   3.069303           12.339867               2.060594  5.861146    0.009896       1.920356       0.002523     761.144613
             MLP 0.207497  0.205558   0.270348            1.495213               0.162505  0.516259    8.490156       1.920356       0.005364     358.039924


In [106]:
mlp = models["MLP"]

y_pred = mlp.predict(X_test)
    mlp.fit(X_train.iloc[:n], y_train[:n])
    predictions = mlp.predict(X_val)
    results.append({
        "training_size": n,
        "val_MAE": mean_absolute_error(y_val, predictions),
        "val_RMSE": np.sqrt(mean_squared_error(y_val, predictions))
    })
learning_results = pd.DataFrame(results)
display(learning_results)

learning_results.to_csv(OUTPUT/"learning_curve_results.csv", index=False)

,training_size,val_MAE,val_RMSE
0,1000,0.458612,0.623832
1,2500,0.385883,0.522211
2,5000,0.261293,0.352022
3,7000,0.240147,0.318923
4,10000,0.220222,0.291977


In [107]:
plt.figure(figsize=(10,10))
plt.scatter(
    y_test,
    abs_errors,
    alpha=0.5,
    s=15
)

plt.xlabel("QuantLib Heston Price")
plt.ylabel("Absolute Pricing Error")
plt.title("MLP Surrogate: Absolute Pricing Error")
plt.grid(alpha=0.3)
plt.show()

NameError: name 'abs_errors' is not defined

<Figure size 720x720 with 0 Axes>

In [ ]:
T_years = data["T_days"] / 365.0
lower_bound = np.maximum(0, S0 - data["K"] * np.exp(-data["r"] * T_years))

valid_prices = (
    np.isfinite(data["price"])
    & (data["price"] >= lower_bound - 1e-5)
    & (data["price"] <= S0 + 1e-5)
)

print("Valid Prices:", valid_prices.sum())
print("Invalid prices:", (~valid_prices).sum())

assert valid_prices.all(), "Invalid option prices found"

Valid Prices: 1000
Invalid prices: 0
